In [1]:
import google.auth


# Resolve credentials and project ID explicitly
credentials, project_id = google.auth.default()
print(f"Project ID: {project_id}")

Project ID: dev-mq-tech-transfer


In [2]:
# Cell 1 — instrumentation (run once per kernel session)
from phoenix.otel import register
from openinference.instrumentation.google_adk import GoogleADKInstrumentor

tracer_provider = register(
    project_name="gap_assessment", # set your project name here when you start the phoenix server
    endpoint="http://localhost:6006/v1/traces"
)
GoogleADKInstrumentor().instrument(tracer_provider=tracer_provider)

c:\Users\L137860\Downloads\DEV WORKSPACE\Gap Assessment\tredence_gap_assessment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OpenTelemetry Tracing Details
|  Phoenix Project: gap_assessment
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [3]:
# Cell 2 — agent definition
import os
from google.adk.agents import Agent

def get_current_time(city: str) -> dict:
    return {"status": "success", "city": city, "time": "10:30 AM"}

root_agent = Agent(
    model=os.environ.get("MODEL_NAME", "gemini-2.5-flash"),
    name='root_agent',
    description="Tells the current time in a specified city.",
    instruction="You are a helpful assistant that tells the current time in cities.",
    tools=[get_current_time],
)

In [4]:
# # Cell 3 — runner
from google.adk.runners import InMemoryRunner
from google.genai.types import Content, Part
# APP_NAME = "time-agent"
# USER_ID = os.environ.get("PHOENIX_USER", "Prakash")


# async def ask(question: str):
#     runner = InMemoryRunner(agent=root_agent, app_name=APP_NAME)
#     session = await runner.session_service.create_session(
#         app_name=APP_NAME, user_id=USER_ID
#     )
#     async for event in runner.run_async(
#         user_id=USER_ID,
#         session_id=session.id,
#         new_message=Content(parts=[Part(text=question)]),
#     ):
#         if event.content and event.content.parts:
#             for part in event.content.parts:
#                 if part.text:
#                     print("Agent:", part.text)

In [5]:
from opentelemetry import trace
APP_NAME = "time-agent"
USER_ID = os.environ.get("PHOENIX_USER", "Prakash")
async def ask(question: str, user: str = os.environ.get("PHOENIX_USER", "unknown")):
    runner = InMemoryRunner(agent=root_agent, app_name=APP_NAME)
    session = await runner.session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID
    )

    # Tag the trace with username
    tracer = trace.get_tracer(__name__)
    with tracer.start_as_current_span("ask") as span:
        span.set_attribute("user.name", user)
        span.set_attribute("question", question)

        async for event in runner.run_async(
            user_id=USER_ID,
            session_id=session.id,
            new_message=Content(parts=[Part(text=question)]),
        ):
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        print("Agent:", part.text)

In [6]:
await ask("What is the time in London?")

c:\Users\L137860\Downloads\DEV WORKSPACE\Gap Assessment\tredence_gap_assessment\.venv\Lib\site-packages\google\adk\tools\function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


Agent: The current time in London is 10:30 AM.


In [7]:
await ask("What is the time in Paris?")

Agent: The current time in Paris is 10:30 AM.
